In [1]:
import lightgbm as lgb
import numpy as np
import polars as pl
from sklearn.model_selection import train_test_split

RANDOM_SEED = 1000

In [2]:
from surrogate_model.data import flatten_features, label, load_and_clean_data

initial_features_file = "data/features/sample_initial_100k_features.parquet"
initial_labels_file = "data/docking_runs/sample_initial_100k.1L83.p2rank.1.parquet"

feature_cols = [
    "atom_pair",
    "autocorr",
    "descriptors",
    "ecfp",
    "functional_groups",
    "morse",
    "rdf",
    "topological_torsion",
    "usrcat",
    "whim",
]

df_clean = load_and_clean_data(initial_features_file, initial_labels_file)

x_initial = flatten_features(df_clean, feature_cols)
y_initial_raw = df_clean["affinity_kcal_mol"]

# "top" 1% (most negative == strongest binder)
y_initial_label, threshold = label(y_initial_raw, percentile=1.0)

In [3]:
x_initial_train, x_initial_test, y_initial_train, y_initial_test = train_test_split(
    x_initial, y_initial_label, train_size=0.8, random_state=RANDOM_SEED
)

model = lgb.LGBMClassifier(
    objective="binary",
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=31,
    min_child_samples=10,
    random_state=RANDOM_SEED,
    deterministic=True,
    force_row_wise=True,
)

model.fit(x_initial_train, y_initial_train)

[LightGBM] [Info] Number of positive: 761, number of negative: 74219
[LightGBM] [Info] Total Bins 211948
[LightGBM] [Info] Number of data points in the train set: 74980, number of used features: 4616
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.010149 -> initscore=-4.580142
[LightGBM] [Info] Start training from score -4.580142


,learning_rate,0.05
,n_estimators,500
,objective,'binary'
,min_child_samples,10
,random_state,1000
,deterministic,True
,force_row_wise,True
,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,subsample_for_bin,200000


In [4]:
from surrogate_model.metrics import evaluation_metrics, predict_active

y_predict_initial = predict_active(model=model, x=x_initial_test)
df_metrics_initial = evaluation_metrics(y_initial_test, y_predict_initial)

In [ ]:
df_metrics_initial

In [5]:
print(df_metrics_initial)

shape: (4, 5)
┌───────┬───────────┬──────────┬───────────────────┬────────────┐
│ Top % ┆ Precision ┆ Recall   ┆ Num active caught ┆ Num active │
│ ---   ┆ ---       ┆ ---      ┆ ---               ┆ ---        │
│ f64   ┆ f64       ┆ f64      ┆ i64               ┆ i64        │
╞═══════╪═══════════╪══════════╪═══════════════════╪════════════╡
│ 1.0   ┆ 0.374332  ┆ 0.384615 ┆ 70                ┆ 182        │
│ 5.0   ┆ 0.160085  ┆ 0.824176 ┆ 150               ┆ 182        │
│ 10.0  ┆ 0.092316  ┆ 0.950549 ┆ 173               ┆ 182        │
│ 15.0  ┆ 0.063678  ┆ 0.983516 ┆ 179               ┆ 182        │
└───────┴───────────┴──────────┴───────────────────┴────────────┘


In [ ]:
holdout_features_file = "data/features/sample_holdout_100k_features.parquet"
holdout_labels_file = "data/docking_runs/sample_holdout_100k.1L83.p2rank.1.parquet"

In [ ]:
df_holdout_clean = load_and_clean_data(holdout_features_file, holdout_labels_file)

x_holdout = flatten_features(df_holdout_clean, feature_cols)
y_holdout_raw = df_holdout_clean["affinity_kcal_mol"]

y_holdout_label, _ = label(y_holdout_raw, threshold=threshold)

y_holdout_evaluation = predict_active(model=model, x=x_holdout)
df_metrics_holdout = evaluation_metrics(y_holdout_label, y_holdout_evaluation)

In [ ]:
df_metrics_holdout

In [ ]:
import glob
import logging

import pyarrow.parquet as pq

logging.basicConfig(
    level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s"
)


def predict_large_parquet(
    file_pattern: str,
    model,
    feature_cols: list[str],
    batch_size: int = 10_000,
    log_interval: int = 100_000,
) -> pl.DataFrame:

    file_paths = glob.glob(file_pattern)
    if not file_paths:
        logging.warning(f"No files found matching pattern '{file_pattern}'")
        return pl.DataFrame({"id": [], "p_active": []})

    total_rows = sum(pq.ParquetFile(f).metadata.num_rows for f in file_paths)
    logging.info(
        f"Starting screening. Found {len(file_paths)} files containing"
        f" {total_rows:,} total molecules."
    )

    results_ids = []
    results_probs = []

    processed_rows = 0
    last_logged_rows = 0

    for file_path in file_paths:
        parquet_file = pq.ParquetFile(file_path)

        for batch in parquet_file.iter_batches(batch_size=batch_size):
            current_batch_size = batch.num_rows
            processed_rows += current_batch_size

            if processed_rows - last_logged_rows >= log_interval:
                percent_complete = (processed_rows / total_rows) * 100
                logging.info(
                    f"Processed {processed_rows:,} / {total_rows:,} rows"
                    f" ({percent_complete:.1f}%)"
                )
                last_logged_rows = processed_rows

            df_batch = pl.from_arrow(batch)
            assert isinstance(df_batch, pl.DataFrame)

            df_clean = df_batch.filter(pl.all_horizontal(pl.col("^.*_valid$")))

            if df_clean.height == 0:
                continue

            x, _, _ = extract_features_and_target(df_clean, feature_cols=feature_cols)

            x_named = (
                x[model.feature_name_]
                if isinstance(x, pl.DataFrame)
                else pl.DataFrame(x, schema=model.feature_name_)
            )

            p_all_classes = np.asarray(model.predict_proba(x_named))
            p_active = p_all_classes[:, 1]

            results_ids.extend(df_clean["id"].to_list())
            results_probs.extend(p_active)

    logging.info(f"Finished! Processed {total_rows:,} / {total_rows:,} rows (100.0%).")
    logging.info(
        f"Successfully generated predictions for {len(results_ids):,} valid molecules."
    )

    df_predictions = pl.DataFrame({"id": results_ids, "p_active": results_probs})

    return df_predictions

In [ ]:
df_preds = predict_large_parquet(
    file_pattern="data/features/*.parquet", model=model, feature_cols=feature_cols
)

In [ ]:
k_1_percent = max(1, int(df_preds.height * 0.01))

df_top_1 = df_preds.sort("p_active", descending=True).head(k_1_percent)

print(f"Total screened: {df_preds.height}")
print(f"Top 1% extracted: {df_top_1.height}")

In [ ]:
def filter_parquet_by_ids(
    source_path: str, ids: list, output_path: str, id_col: str = "id"
) -> None:
    (
        pl.scan_parquet(source_path)
        .filter(pl.col(id_col).is_in(ids))
        .sink_parquet(output_path)
    )

In [ ]:
n_samples = min(100_000, df_top_1.height)
sampled_ids = df_top_1["id"].sample(n=n_samples, seed=RANDOM_SEED).to_list()

In [ ]:
filter_parquet_by_ids(
    source_path="data/conformers/shard_*",
    ids=sampled_ids,
    output_path="data/conformers/active_round_1.parquet",
)

In [ ]:
filter_parquet_by_ids(
    source_path="data/features/shard_*",
    ids=sampled_ids,
    output_path="data/features/active_round_1.parquet",
)

In [ ]:
active_round_1_features_file = "data/features/active_round_1.parquet"
active_round_1_labels_file = "data/docking_runs/active_round_1.1L83.p2rank.1.parquet"

df_active_round_1_clean = load_and_clean_data(
    active_round_1_features_file, active_round_1_labels_file
)
x_active_round_1, y_active_round_1, feature_names = extract_features_and_target(
    df_active_round_1_clean, feature_cols, "affinity_kcal_mol"
)
y_active_round_1_binary = (y_active_round_1 <= active_threshold).astype(int)

In [ ]:
(
    x_active_round_1_train,
    x_active_round_1_test,
    y_active_round_1_train,
    y_active_round_1_test,
) = train_test_split(
    np.concatenate((x_initial, x_active_round_1)),
    np.concatenate((y_initial_binary, y_active_round_1_binary)),
    train_size=0.8,
    random_state=RANDOM_SEED,
)

In [ ]:
model_active = lgb.LGBMClassifier(
    objective="binary",
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=31,
    min_child_samples=10,
    random_state=RANDOM_SEED,
    deterministic=True,
    force_row_wise=True,
)

model_active.fit(
    x_active_round_1_train, y_active_round_1_train, feature_name=feature_names
)

In [ ]:
y_active_round_1_test_binary, p_active_round_1_test, k_active_round_1_test = (
    prepare_evaluation_data(
        model=model_active,
        x=x_active_round_1_test,
        y=y_active_round_1_test,
        active_threshold=active_threshold,
    )
)

df_active_round_1_metrics = evaluate_enrichment(
    y_active_round_1_test, p_active_round_1_test
)

In [ ]:
y_holdout_active_round_1_binary, p_holdout_active_round_1, k_holdout_active_round_1 = (
    prepare_evaluation_data(
        model=model_active,
        x=x_holdout,
        y=y_holdout_label,
        active_threshold=active_threshold,
    )
)
df_holdout_active_round_1_metrics = evaluate_enrichment(
    y_holdout_active_round_1_binary,
    p_holdout_active_round_1,
    [0.01, 0.02, 0.03, 0.04, 0.05, 0.1],
)